# Ship the churn model: three runs, one alias, one endpoint, one batch

One row is one telco customer; the pipeline is D01 5.6's, and the threshold is 2.2's 0.278. You log three models to MLflow, register the best one under the alias `champion`, load it back by that alias, save it for a service, then serve it and score a batch from the terminal.

This notebook works inside the activity folder, `D03/05-Evr-Ship-Churn-Model`: MLflow's records (`mlflow.db`, `mlruns/`) and `churn_model.joblib` are written there, beside `app.py`, `score_batch.py`, `customer.json` and `customers.csv`. To start again from scratch, delete `mlflow.db`, `mlruns/` and `churn_model.joblib` in that folder.

## The starter

The cell moves into the activity folder, loads the data, and rebuilds D01 5.6's preparation and split. `MLFLOW_DISABLE_TELEMETRY` stops MLflow from sending usage data.

In [ ]:
import os
import warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
# MLflow 3.16's autolog calls a scikit-learn metric with an argument name that scikit-learn 1.9 renamed,
# and it warns that the input example has integer columns. Both warnings concern the two libraries, not
# your code, so these lines hide them.
warnings.filterwarnings("ignore", message="`y_pred` was renamed")
warnings.filterwarnings("ignore", message="Hint: Inferred schema contains integer")
here = Path.cwd()
if here.name in ("Unsolved", "Solved"):
    os.chdir("..")                                   # the notebook was started in its own folder
elif here.name != "05-Evr-Ship-Churn-Model":
    os.chdir("D03/05-Evr-Ship-Churn-Model")          # VS Code starts notebooks at the repo root
print("working in", Path.cwd().name)

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../../data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)


def make_pre(scale=True):
    """D01 5.6's ColumnTransformer; scale=False for the tree models."""
    num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if scale else SimpleImputer(strategy="median")
    return ColumnTransformer([("num", num, num_cols),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


cv = StratifiedKFold(5, shuffle=True, random_state=42)
THRESHOLD = 0.278
print(len(X_train), "training customers,", len(X_test), "test customers")

## Step 1 · Log three runs

Each run fits one model inside `mlflow.start_run(run_name=...)`, logs its 5-fold CV ROC AUC with `mlflow.log_metric("cv_auc", ...)`, and saves the fitted pipeline with `mlflow.sklearn.log_model`. `mlflow.sklearn.autolog(log_models=False)` records every setting for you.

MLflow 3.16 saves scikit-learn models in the skops format, which refuses to load types it was not told to trust. `skops_trusted_types` lists them, so the model can be loaded back in step 4.

In [ ]:
import logging

import mlflow
import skops.io as sio
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logging.getLogger("mlflow").setLevel(logging.ERROR)     # MLflow logs every step it takes; keep only errors
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("telco-churn")
mlflow.sklearn.autolog(log_models=False, silent=True)

candidates = {
    "logistic": make_pipeline(make_pre(), LogisticRegression(max_iter=1000)),
    "forest": make_pipeline(make_pre(scale=False), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    "boosted": make_pipeline(make_pre(scale=False), HistGradientBoostingClassifier(random_state=42)),
}
model_uris = {}
for name, model in candidates.items():
    with mlflow.start_run(run_name=name):
        cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
        model.fit(X_train, y_train)
        mlflow.log_metric("cv_auc", float(cv_auc.mean()))
        trusted = sio.get_untrusted_types(data=sio.dumps(model))
        info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),
                                        skops_trusted_types=trusted)
        model_uris[name] = info.model_uri
    print(f"{name:<9} cv_auc {cv_auc.mean():.3f}")

## Step 2 · Compare the runs

`mlflow.search_runs(order_by=["metrics.cv_auc DESC"])[["tags.mlflow.runName", "metrics.cv_auc"]]`

## Step 3 · Register the best run and give it an alias

`mv = mlflow.register_model(uri_of_best, "telco-churn")`, then `MlflowClient().set_registered_model_alias("telco-churn", "champion", mv.version)`. The URI of each run's model is in `model_uris`.

## Step 4 · Load it by alias, the way a service would

`champion = mlflow.sklearn.load_model("models:/telco-churn@champion")`. Score the test customers and print the ROC AUC.

## Step 5 · Save the pipeline for the service

`joblib.dump(champion, "churn_model.joblib")`. The file lands in the activity folder, where `app.py` and `score_batch.py` look for it.

## Steps 6 to 9 · In the terminal

From here on you work in the VS Code terminal, inside the activity folder (`cd D03/05-Evr-Ship-Churn-Model`), as the README describes.

6. Start the endpoint in a second terminal and leave it running: `uvicorn app:app --port 8000`
7. Call it from the first terminal: `curl -X POST localhost:8000/predict -H "Content-Type: application/json" -d @customer.json`
8. Delete the `tenure` line from a copy of `customer.json`, call again, and read the status code and the message.
9. Score a batch: `python score_batch.py customers.csv scores.csv --model churn_model.joblib --threshold 0.278`, then open `scores.csv`.

## Steps 6 to 9 from Python

The same calls, made from this notebook so it can check itself: start the endpoint as a background process, call it with `urllib` (what `curl` does), stop it, then run the batch script.

## Bonus · A challenger

Register the `class_weight="balanced"` model as version 2 with the alias `challenger`, load it by alias, and compare its test recall at the 0.5 default with the champion's.

## Check yourself